# Görev 3: Mini-Beyin (nn.Module ve Sınıf Yapısı)

Bu görevde amacımız PyTorch modellerini nesne yönelimli (OOP) olarak nasıl yazdığımızı
görmek. `nn.Module`'den miras alan `MiniBrain` adında, 2 gizli katmanlı basit bir MLP
(Çok Katmanlı Algılayıcı) yazdık. Bunu neden sınıf olarak yazdık? Çünkü Transformer gibi
çok daha karmaşık mimariler bile aslında hep bu mantıkla, küçük sınıflardan kurulu -
o yüzden bu basit örnekte de aynı kalıbı kullanmak istedik.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)


class MiniBrain(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        # görevde istenen boyutlarda 3 tane Linear katman: giriş -> 128 -> 64 -> çıkış
        self.fc1 = nn.Linear(input_size, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, output_size)

    def forward(self, x):
        # veri sırayla katmanlardan geçiyor, gizli katmanlardan sonra relu uyguluyoruz
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)  # son katmanda relu yok, çıktıyı sınırlamak istemedik
        return x

## Modeli oluşturup mimariyi yazdıralım

`input_size=10`, `output_size=3` seçtik (görev özel bir sayı vermediği için kendimiz
belirledik). `print(model)` dediğimizde PyTorch otomatik olarak katmanları ve
boyutlarını güzelce listeliyor, elle bir şey yazmamıza gerek kalmıyor.

In [2]:
model = MiniBrain(input_size=10, output_size=3)
print(model)

MiniBrain(
  (fc1): Linear(in_features=10, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=3, bias=True)
)


## Test: rastgele bir girdi verip çalışıp çalışmadığına bakalım

`(batch_size, input_size)` yani `(4, 10)` boyutunda rastgele bir tensör oluşturup
modele verdik. Boyut hatası almadan `(4, 3)` boyutunda bir çıktı gelirse mimarinin
doğru kurulduğunu anlamış oluruz.

In [3]:
batch_size = 4
input_size = 10
x = torch.randn(batch_size, input_size)
print("girdi shape:", x.shape)

out = model(x)
print("çıktı shape:", out.shape)
print("çıktı:\n", out)

girdi shape: torch.Size([4, 10])
çıktı shape: torch.Size([4, 3])
çıktı:
 tensor([[-0.0087, -0.0724,  0.0059],
        [-0.1751, -0.0611,  0.0072],
        [-0.0952, -0.1358,  0.0553],
        [-0.0515, -0.1031,  0.0207]], grad_fn=<AddmmBackward0>)


## Değerlendirme: `forward` metodu neden bu kadar önemli?

`__init__` içinde sadece hangi katmanların var olduğunu tanımlıyoruz, ama bunların
birbirine nasıl bağlanacağını, hangi sırayla çalışacağını ve aralarında hangi
aktivasyon fonksiyonunun kullanılacağını `forward` metodu belirliyor. Yani `__init__`
malzemeleri hazırlıyor, `forward` ise bu malzemelerle asıl "tarifi" yazıyor diyebiliriz.
PyTorch, `model(x)` çağırdığımızda arka planda otomatik olarak bu `forward` metodunu
çalıştırıyor, hem de gradyan hesabı için gereken hesap grafiğini (computation graph)
kendisi kuruyor. Bu yüzden mimari ne kadar karmaşık olursa olsun (Transformer'lar
dahil), asıl mantık hep `forward` içinde aranır - katmanlar sadece birer alet çantası,
gerçek akışı belirleyen odur.